# Section 14: Post-Review Revision of the Final Adaptive Rule

Review of the first-pass results showed that the selected rule (A3, contention-only) produced a mean selected threshold of 0.4995 and a maximum of exactly 0.50, so it behaved almost identically to a fixed threshold. The contention-increase term compares each request's current contention with its starting value, and contention is normalised by the shrinking set of valid requests. As scheduling proceeds the ratio rarely rises above its initial value, so the term is close to zero for most of an episode.

The A3 and A6 finalists were also statistically tied on W40 (difference of about 0.00002 in duration fulfillment, with the sign changing across training seeds). The tie-break rule below is declared before it is applied and uses W40 evidence only. W50 and the cross-week results play no part in the choice.

**Declared tie-break rule:** if the two highest-ranked finalists differ in mean duration fulfillment by less than the larger of their across-training-seed standard deviations, they are treated as tied, and the finalist whose selected thresholds are reduced further below the base threshold (the rule that actually adapts) is selected.

The first-pass A3 outputs are archived before anything is overwritten.

## 14.1 Archive the First-Pass Outputs

In [ ]:
import shutil

ARCHIVE_DIR = DIRS["results"] / "archive_a3_first_pass"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

FIRST_PASS_FILES = [
    DIRS["results"] / "adaptive_cross_week_results.csv",
    DIRS["results"] / "adaptive_cross_week_summary.csv",
    DIRS["results"] / "adaptive_model_level_summary.csv",
    DIRS["results"] / "adaptive_overall_summary.csv",
    DIRS["results"] / "adaptive_training_runs.csv",
    DIRS["results"] / "adaptive_threshold_results.csv",
    DIRS["results"] / "adaptive_paired_model_comparison.csv",
    DIRS["configs"] / "adaptive_bee_configuration.json",
    DIRS["report"] / "adaptive_bee_report_section.md",
]

archived = []

for path in FIRST_PASS_FILES:
    target = ARCHIVE_DIR / path.name
    if path.exists() and not target.exists():
        shutil.copy2(path, target)
        archived.append(path.name)

figure_archive = ARCHIVE_DIR / "figures"
figure_archive.mkdir(exist_ok=True)

for figure_path in DIRS["figures"].glob("*.png"):
    target = figure_archive / figure_path.name
    if not target.exists():
        shutil.copy2(figure_path, target)

print("Archive folder:", ARCHIVE_DIR)
print("Newly archived files:", archived)

## 14.2 Diagnose Which Adaptive Terms Move the Threshold

A random-policy episode on W40 is run with the reference rule (A1), where all three terms are active. For every step, the mean contribution of each term to the threshold reduction is recorded. If the contention term stays near zero until the final steps while the scarcity and unmet-demand terms rise, the first-pass behavior of A3 is explained.

In [ ]:
diagnostic_environment = create_adaptive_environment(
    ["W40_2018"], SEED, ADAPTIVE_CONFIGURATIONS["A1"], True
)
diagnostic_environment.reset(seed=SEED, options={"week_key": "W40_2018"})

diagnostic_rng = np.random.default_rng(0)
diagnostic_rows = []

while True:
    valid_actions = np.flatnonzero(diagnostic_environment.action_masks())

    if len(valid_actions) == 0:
        break

    rows = [diagnostic_environment.bee_feature_cache[int(a)] for a in valid_actions]

    diagnostic_rows.append(
        {
            "step": int(diagnostic_environment.step_count),
            "valid_requests": int(len(valid_actions)),
            "scarcity_term": 0.20 * float(np.mean([r["placement_scarcity"] for r in rows])),
            "contention_term": 0.15 * float(np.mean([r["contention_increase"] for r in rows])),
            "unmet_term": 0.15 * float(np.mean([r["relative_unmet_pressure"] for r in rows])),
            "mean_threshold": float(np.mean([r["threshold"] for r in rows])),
            "minimum_threshold": float(np.min([r["threshold"] for r in rows])),
        }
    )

    _, _, terminated, truncated, _ = diagnostic_environment.step(
        int(diagnostic_rng.choice(valid_actions))
    )

    if terminated or truncated:
        break

threshold_diagnostic = pd.DataFrame(diagnostic_rows)
threshold_diagnostic.to_csv(DIRS["results"] / "adaptive_threshold_diagnostic.csv", index=False)

display(threshold_diagnostic.iloc[::10].round(4))
print(threshold_diagnostic[["scarcity_term", "contention_term", "unmet_term"]].describe().round(4))

plt.figure(figsize=(10, 5))

for column, label in [
    ("scarcity_term", "Scarcity term (0.20 x scarcity)"),
    ("contention_term", "Contention term (0.15 x contention increase)"),
    ("unmet_term", "Unmet term (0.15 x relative unmet pressure)"),
]:
    plt.plot(threshold_diagnostic["step"], threshold_diagnostic[column], label=label)

plt.xlabel("Scheduling step")
plt.ylabel("Mean threshold reduction")
plt.legend()
save_figure("adaptive_threshold_terms_during_episode.png")

## 14.3 Apply the Declared Tie-Break Rule Using W40 Evidence Only

In [ ]:
ACTIVITY_MINIMUM_REDUCTION = 0.005
BASE_THRESHOLD = COMMON_ADAPTIVE_PARAMETERS["base_threshold"]

search_results = pd.read_csv(DIRS["results"] / "adaptive_configuration_search.csv")
finalists = search_results[search_results["stage"] == "confirmation"]

finalist_models = (
    finalists.groupby(["configuration_id", "training_seed"], as_index=False)[
        ["duration_fulfillment", "mean_selected_threshold", "minimum_selected_threshold"]
    ].mean()
)

finalist_summary = (
    finalist_models.groupby("configuration_id")
    .agg(
        fulfillment_mean=("duration_fulfillment", "mean"),
        fulfillment_seed_std=("duration_fulfillment", "std"),
        mean_threshold=("mean_selected_threshold", "mean"),
        minimum_threshold=("minimum_selected_threshold", "mean"),
    )
    .reset_index()
    .sort_values("fulfillment_mean", ascending=False)
    .reset_index(drop=True)
)

finalist_summary["mean_threshold_reduction"] = BASE_THRESHOLD - finalist_summary["mean_threshold"]
display(finalist_summary.round(5))

top, runner_up = finalist_summary.iloc[0], finalist_summary.iloc[1]
gap = float(top["fulfillment_mean"] - runner_up["fulfillment_mean"])
noise = float(max(top["fulfillment_seed_std"], runner_up["fulfillment_seed_std"]))
finalists_tied = gap < noise

if finalists_tied:
    chosen = finalist_summary.sort_values("mean_threshold_reduction", ascending=False).iloc[0]
    tie_break_applied = True
else:
    chosen = top
    tie_break_applied = False

SELECTED_CONFIGURATION_ID = str(chosen["configuration_id"])
SELECTED_CONFIGURATION = ADAPTIVE_CONFIGURATIONS[SELECTED_CONFIGURATION_ID]

print(f"Fulfillment gap between finalists: {gap:.6f}")
print(f"Larger across-seed standard deviation: {noise:.6f}")
print("Finalists tied:", finalists_tied)
print("Selected configuration:", SELECTED_CONFIGURATION_ID)
print(f"Mean threshold reduction of selected rule: {float(chosen['mean_threshold_reduction']):.4f}")

if float(chosen["mean_threshold_reduction"]) < ACTIVITY_MINIMUM_REDUCTION:
    print("WARNING: the selected rule barely adapts. Review the diagnostic before retraining.")

## 14.4 Re-Freeze the Final Configuration

The revised configuration is written to the same JSON path. The first-pass file remains in the archive folder.

In [ ]:
with open(ARCHIVE_DIR / "adaptive_bee_configuration.json", "r") as file:
    first_pass_configuration = json.load(file)

revised_configuration = dict(first_pass_configuration)

revised_configuration.update(
    {
        "selected_configuration_id": SELECTED_CONFIGURATION_ID,
        "adaptive_parameters": SELECTED_CONFIGURATION,
        "selection_reason": (
            "Finalists were statistically tied on W40, so the declared tie-break rule "
            "selected the finalist whose thresholds adapt further below the base threshold."
            if tie_break_applied
            else "Highest lexicographic validation ranking on W40 after staged screening and three-seed confirmation."
        ),
        "revision": {
            "first_pass_selected_configuration_id": first_pass_configuration["selected_configuration_id"],
            "first_pass_mean_threshold_reduction": float(
                finalist_summary.loc[
                    finalist_summary["configuration_id"] == first_pass_configuration["selected_configuration_id"],
                    "mean_threshold_reduction",
                ].iloc[0]
            ),
            "fulfillment_gap_between_finalists": gap,
            "larger_across_seed_std": noise,
            "tie_break_applied": bool(tie_break_applied),
            "tie_break_rule": "If the fulfillment gap is below the larger across-seed standard deviation, select the finalist with the larger mean threshold reduction.",
            "evidence_used": "W40 validation only",
        },
    }
)

with open(FROZEN_CONFIG_PATH, "w") as file:
    json.dump(to_builtin(revised_configuration), file, indent=2)

print("Revised configuration saved:", FROZEN_CONFIG_PATH)
print("Selected:", SELECTED_CONFIGURATION_ID)

## 14.5 Revised Logic Tests

Every adaptive configuration is now tested across a complete episode rather than at a single step. The tests verify the stimulus, threshold and activation equations against the stored feature values, the selective-adaptation gate for A6, the threshold bounds, activation range, the starting thresholds, and that thresholds never change the physical action mask compared with the fixed-threshold environment.

In [ ]:
from bee_fixed_environment import BEEFixedThresholdEnv


def build_fixed_reference_environment(week_key, shuffle_requests=False):
    requests, visibility, maintenance = select_weeks(
        request_table, visibility_table, maintenance_table, [week_key]
    )

    return BEEFixedThresholdEnv(
        requests=requests,
        visibility_periods=visibility,
        maintenance=maintenance,
        seed=SEED,
        max_requests=MAX_REQUESTS,
        max_steps=MAX_STEPS,
        shuffle_requests=shuffle_requests,
        stimulus_weights=tuple(COMMON_ADAPTIVE_PARAMETERS["stimulus_weights"]),
        fixed_threshold=COMMON_ADAPTIVE_PARAMETERS["base_threshold"],
        sensitivity=COMMON_ADAPTIVE_PARAMETERS["sensitivity"],
    )


def verify_features(environment, configuration):
    weights = configuration["stimulus_weights"]
    base = configuration["base_threshold"]
    sensitivity = configuration["sensitivity"]
    observed_thresholds = []

    for action, bee in environment.bee_feature_cache.items():
        manual_stimulus = np.clip(
            weights[0] * bee["placement_scarcity"]
            + weights[1] * bee["resource_contention"]
            + weights[2] * bee["mission_unmet_ratio"],
            0.0,
            1.0,
        )

        pressure = (
            int(bee["placement_scarcity"] >= configuration["scarcity_indicator_limit"])
            + int(bee["contention_increase"] >= configuration["contention_indicator_limit"])
            + int(bee["relative_unmet_pressure"] >= configuration["unmet_indicator_limit"])
        )

        enabled = (not configuration["selective_mode"]) or pressure >= configuration["minimum_pressure_indicators"]

        if enabled:
            manual_threshold = np.clip(
                base
                - configuration["scarcity_strength"] * bee["placement_scarcity"]
                - configuration["contention_strength"] * bee["contention_increase"]
                - configuration["unmet_strength"] * bee["relative_unmet_pressure"],
                configuration["minimum_threshold"],
                configuration["maximum_threshold"],
            )
        else:
            manual_threshold = base

        manual_activation = manual_stimulus ** sensitivity / (
            manual_stimulus ** sensitivity + manual_threshold ** sensitivity + 1e-8
        )

        assert np.isclose(bee["stimulus"], manual_stimulus), (action, "stimulus")
        assert bee["pressure_count"] == pressure, (action, "pressure_count")
        assert bee["adaptation_enabled"] == enabled, (action, "adaptation_enabled")
        assert np.isclose(bee["threshold"], manual_threshold), (action, "threshold")
        assert np.isclose(bee["activation"], manual_activation), (action, "activation")
        assert configuration["minimum_threshold"] <= bee["threshold"] <= configuration["maximum_threshold"]
        assert bee["threshold"] <= base + 1e-12
        assert 0.0 <= bee["activation"] <= 1.0

        observed_thresholds.append(bee["threshold"])

    return observed_thresholds


logic_test_rows = []

for configuration_id, configuration in ADAPTIVE_CONFIGURATIONS.items():
    adaptive_environment = create_adaptive_environment(["W10_2018"], SEED, configuration, False)
    fixed_environment = build_fixed_reference_environment("W10_2018", False)

    adaptive_environment.reset(seed=SEED, options={"week_key": "W10_2018"})
    fixed_environment.reset(seed=SEED, options={"week_key": "W10_2018"})

    assert np.allclose(
        [row["threshold"] for row in adaptive_environment.bee_feature_cache.values()],
        configuration["base_threshold"],
    ), "Thresholds must start at the base threshold."

    all_thresholds = []
    steps_checked = 0

    while True:
        assert np.array_equal(adaptive_environment.action_masks(), fixed_environment.action_masks()), (
            configuration_id, steps_checked, "physical mask changed"
        )

        all_thresholds.extend(verify_features(adaptive_environment, configuration))

        valid_actions = np.flatnonzero(fixed_environment.action_masks())

        if len(valid_actions) == 0:
            break

        action = int(valid_actions[0])
        _, _, adaptive_done, adaptive_truncated, _ = adaptive_environment.step(action)
        _, _, fixed_done, fixed_truncated, _ = fixed_environment.step(action)
        steps_checked += 1

        if adaptive_done or adaptive_truncated or fixed_done or fixed_truncated:
            break

    logic_test_rows.append(
        {
            "configuration_id": configuration_id,
            "steps_checked": steps_checked,
            "minimum_threshold_seen": float(np.min(all_thresholds)),
            "mean_threshold_seen": float(np.mean(all_thresholds)),
            "maximum_threshold_seen": float(np.max(all_thresholds)),
        }
    )

display(pd.DataFrame(logic_test_rows).round(4))
print("Equation, selective-gate, bound, activation-range, base-start and mask-independence tests passed for A1 to A6.")

# 14.6 Retrain the Revised Rule Across the Five Held-Out Folds

The revised rule is retrained from scratch for every fold with the same PPO configuration, reward, 25,088-timestep budget, training seeds and request-order seeds as before. Run identifiers carry the configuration name, so the archived first-pass models are never reused. Every result is saved immediately and the cell is resumable after a disconnection.

In [ ]:
cross_week_results = []
training_run_records = []

for fold_definition in ROBUSTNESS_FOLDS:
    fold = int(fold_definition["fold"])
    training_weeks = fold_definition["training_weeks"]
    test_week = fold_definition["test_week"]

    fold_training_seeds = [42, 123, 456] if test_week == "W50_2018" else [42]

    for training_seed in fold_training_seeds:
        run_id = f"{SELECTED_CONFIGURATION_ID.lower()}_fold_{fold}_{test_week.lower()}"

        model_path, metadata = train_adaptive_model(
            run_type="cross_week",
            run_id=run_id,
            training_weeks=training_weeks,
            training_seed=training_seed,
            configuration_id=SELECTED_CONFIGURATION_ID,
            adaptive_configuration=SELECTED_CONFIGURATION,
            target_timesteps=CONTROLLED_TIMESTEPS,
        )

        metadata["fold"] = fold
        metadata["test_week"] = test_week
        training_run_records.append(metadata)

        for evaluation_seed in EVALUATION_SEEDS:
            result = evaluate_adaptive_model(
                model_path=model_path,
                run_type="cross_week",
                run_id=run_id,
                configuration_id=SELECTED_CONFIGURATION_ID,
                adaptive_configuration=SELECTED_CONFIGURATION,
                evaluation_week=test_week,
                training_weeks=training_weeks,
                training_seed=training_seed,
                evaluation_seed=evaluation_seed,
                fold=fold,
            )
            cross_week_results.append(result)

adaptive_training_runs = pd.DataFrame(training_run_records)
adaptive_cross_week_results = pd.DataFrame(cross_week_results)

adaptive_training_runs.to_csv(DIRS["results"] / "adaptive_training_runs.csv", index=False)
adaptive_cross_week_results.to_csv(DIRS["results"] / "adaptive_cross_week_results.csv", index=False)

print("Training runs:", len(adaptive_training_runs))
print("Evaluation runs:", len(adaptive_cross_week_results))

## 14.7 Verify Completeness and Constraint Validity

In [ ]:
assert len(adaptive_training_runs) == 7
assert len(adaptive_cross_week_results) == 7 * len(EVALUATION_SEEDS)
assert adaptive_cross_week_results["constraint_violations"].eq(0).all()
assert set(adaptive_cross_week_results["configuration_id"].unique()) == {SELECTED_CONFIGURATION_ID}
assert set(adaptive_cross_week_results["test_week"].unique()) == set(
    ["W10_2018", "W20_2018", "W30_2018", "W40_2018", "W50_2018"]
)

for metric in [
    "duration_fulfillment",
    "minimum_satisfaction_rate",
    "full_satisfaction_rate",
    "mission_fairness",
    "mean_selected_stimulus",
    "mean_selected_threshold",
    "minimum_selected_threshold",
    "maximum_selected_threshold",
    "mean_selected_activation",
]:
    assert adaptive_cross_week_results[metric].between(0, 1).all(), metric

print("All revised cross-week experiments are complete and valid.")

## 14.8 Re-Aggregate Results and Export Threshold Behavior

In [ ]:
model_level_summary = (
    adaptive_cross_week_results.groupby(["fold", "test_week", "training_seed"], as_index=False)[AGGREGATION_METRICS].mean()
)

fold_level_summary = (
    model_level_summary.groupby(["fold", "test_week"], as_index=False)[AGGREGATION_METRICS].mean()
)

overall_summary = pd.DataFrame(
    {
        "method": ["adaptive_bee"],
        **{metric: [fold_level_summary[metric].mean()] for metric in AGGREGATION_METRICS},
    }
)

adaptive_threshold_results = fold_level_summary[
    [
        "fold",
        "test_week",
        "mean_selected_threshold",
        "minimum_selected_threshold",
        "maximum_selected_threshold",
        "mean_selected_stimulus",
        "mean_selected_activation",
    ]
].copy()

model_level_summary.to_csv(DIRS["results"] / "adaptive_model_level_summary.csv", index=False)
fold_level_summary.to_csv(DIRS["results"] / "adaptive_cross_week_summary.csv", index=False)
overall_summary.to_csv(DIRS["results"] / "adaptive_overall_summary.csv", index=False)
adaptive_threshold_results.to_csv(DIRS["results"] / "adaptive_threshold_results.csv", index=False)

display(fold_level_summary)
display(overall_summary)
display(adaptive_threshold_results)

first_pass_summary = pd.read_csv(ARCHIVE_DIR / "adaptive_cross_week_summary.csv")
comparison_with_first_pass = fold_level_summary[["fold", "test_week", "duration_fulfillment", "mean_selected_threshold"]].merge(
    first_pass_summary[["fold", "duration_fulfillment", "mean_selected_threshold"]],
    on="fold",
    suffixes=("_revised", "_first_pass"),
)
display(comparison_with_first_pass.round(4))

## 14.9 Regenerate the Figures

In [ ]:
trace_frames = []

for row in adaptive_cross_week_results.itertuples():
    trace = pd.read_parquet(row.threshold_path)
    trace["fold"] = row.fold
    trace["test_week"] = row.test_week
    trace["training_seed"] = row.training_seed
    trace["evaluation_seed"] = row.evaluation_seed
    trace_frames.append(trace)

all_traces = pd.concat(trace_frames, ignore_index=True)

search_results = pd.read_csv(DIRS["results"] / "adaptive_configuration_search.csv")
finalist_models = (
    search_results[search_results["stage"] == "confirmation"]
    .groupby(["configuration_id", "training_seed"], as_index=False)["duration_fulfillment"].mean()
)

# Validation performance of the finalists across training seeds (W40)
plt.figure(figsize=(7, 5))
sns.barplot(data=finalist_models, x="configuration_id", y="duration_fulfillment", errorbar="sd", color=ADAPTIVE_COLOR)
sns.stripplot(data=finalist_models, x="configuration_id", y="duration_fulfillment", color="black", size=6)
plt.xlabel("Adaptive configuration")
plt.ylabel("Duration fulfillment")
plt.ylim(0.50, 0.58)
save_figure("adaptive_validation_by_rule.png")

# Stage 1 screening of all six rules
screening = search_results[search_results["stage"] == "screening"]

plt.figure(figsize=(9, 5))
sns.barplot(data=screening, x="configuration_id", y="duration_fulfillment", errorbar="sd", color=ADAPTIVE_COLOR)
plt.xlabel("Adaptive configuration")
plt.ylabel("Duration fulfillment")
plt.ylim(0.50, 0.58)
save_figure("adaptive_screening_by_rule.png")

# Cross-week duration fulfillment with individual trained models
plt.figure(figsize=(9, 5))
sns.barplot(data=fold_level_summary, x="test_week", y="duration_fulfillment", color=ADAPTIVE_COLOR)
sns.stripplot(data=model_level_summary, x="test_week", y="duration_fulfillment", color="black", size=6)
plt.xlabel("Held-out scheduling week")
plt.ylabel("Duration fulfillment")
plt.ylim(0, 1)
save_figure("adaptive_cross_week_fulfillment.png")

# Mean and minimum selected threshold by week
threshold_by_week = fold_level_summary.melt(
    id_vars="test_week",
    value_vars=["mean_selected_threshold", "minimum_selected_threshold"],
    var_name="statistic",
    value_name="threshold",
)

plt.figure(figsize=(9, 5))
sns.pointplot(data=threshold_by_week, x="test_week", y="threshold", hue="statistic", linestyles="none", markers=["o", "s"])
plt.axhline(BASE_THRESHOLD, color="gray", linestyle="--", linewidth=1)
plt.xlabel("Held-out scheduling week")
plt.ylabel("Selected threshold")
plt.ylim(0.1, 0.55)
plt.legend(title="")
save_figure("adaptive_mean_threshold_by_week.png")

# Threshold distribution over every selected request in every cross-week run
plt.figure(figsize=(9, 5))
sns.histplot(data=all_traces, x="threshold", bins=40, color=ADAPTIVE_COLOR)
plt.yscale("log")
plt.xlabel("Selected adaptive threshold")
plt.ylabel("Selected requests (log scale)")
save_figure("adaptive_threshold_distribution.png")

# Threshold change during an episode, averaged over all runs
plt.figure(figsize=(10, 5))
sns.lineplot(data=all_traces, x="step", y="threshold", errorbar="sd", color=ADAPTIVE_COLOR)
plt.axhline(BASE_THRESHOLD, color="gray", linestyle="--", linewidth=1)
plt.xlabel("Scheduling step")
plt.ylabel("Selected adaptive threshold")
plt.ylim(0.1, 0.55)
save_figure("adaptive_threshold_during_episode.png")

# Stimulus versus activation, coloured by threshold
sampled_traces = all_traces.sample(n=min(5000, len(all_traces)), random_state=SEED)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(sampled_traces["stimulus"], sampled_traces["activation"], c=sampled_traces["threshold"], cmap="viridis", s=14)
plt.colorbar(scatter, label="Threshold")
plt.xlabel("Stimulus")
plt.ylabel("Activation")
plt.xlim(0, 1)
plt.ylim(0, 1)
save_figure("adaptive_stimulus_vs_activation.png")

## 14.10 Revised Figure Captions and Report Revision Note

In [ ]:
figure_captions = {
    "adaptive_validation_by_rule.png": "Duration fulfillment of the two finalist adaptive rules on validation week W40 after confirmation with three training seeds. Bars show the mean across training seeds, dots show individual trained models and error bars show one standard deviation. The vertical axis is truncated at 0.50.",
    "adaptive_screening_by_rule.png": "Stage 1 screening of the six adaptive rules on W40 using one model per rule trained for 1,024 timesteps. Error bars show variation across request-order seeds. Differences between rules are smaller than this variation. The vertical axis is truncated at 0.50.",
    "adaptive_cross_week_fulfillment.png": "Duration fulfillment of the revised adaptive BEE model on each held-out week. Bars show the fold mean and dots show individual trained models; only W50 has more than one trained model.",
    "adaptive_mean_threshold_by_week.png": "Mean and minimum selected-request threshold by held-out week. The dashed line marks the base threshold of 0.50 and the vertical axis is truncated.",
    "adaptive_threshold_distribution.png": "Distribution of thresholds associated with every selected request across all cross-week evaluation runs (log-scale counts).",
    "adaptive_threshold_during_episode.png": "Selected-request threshold over the course of schedule construction, averaged over all cross-week evaluation runs. The shaded band shows one standard deviation and the dashed line marks the base threshold.",
    "adaptive_stimulus_vs_activation.png": "Stimulus against activation for a random sample of selected requests across all cross-week runs, coloured by the adaptive threshold at selection.",
    "adaptive_threshold_terms_during_episode.png": "Mean threshold reduction contributed by each adaptive term during a random-policy episode on W40 under the reference rule A1.",
}

with open(DIRS["report"] / "adaptive_figure_captions.json", "w") as file:
    json.dump(figure_captions, file, indent=2)

print("Captions saved.")

In [ ]:
# Re-run the report cell in Section 12 first, then run this cell to append the revision note once.
REPORT_PATH = DIRS["report"] / "adaptive_bee_report_section.md"
REVISION_MARKER = "## Revision of the Final Adaptive Rule"

overall = overall_summary.iloc[0]

revision_note = f"""

{REVISION_MARKER}

The first-pass selection chose A3 (contention-only adaptation). After review, its mean selected threshold was found to be {float(first_pass_summary['mean_selected_threshold'].mean()):.4f}, with a maximum of exactly {BASE_THRESHOLD:.2f}, so it behaved almost like a fixed threshold. The contention-increase term compares current contention with its starting value and is normalised by the shrinking set of valid requests, so it seldom rises above zero during an episode. The two finalists were also tied on W40 (fulfillment gap {gap:.5f}, below the across-seed standard deviation of {noise:.5f}).

A tie-break rule was declared before it was applied and used W40 evidence only: among tied finalists, select the rule whose thresholds adapt further below the base threshold. This selected {SELECTED_CONFIGURATION_ID}. The model was then retrained across all five folds with the same PPO configuration, reward, budget, training seeds and request-order seeds. The first-pass A3 results are retained in the archive folder for transparency. Overall duration fulfillment for the revised model is {float(overall['duration_fulfillment']):.4f} and the mean selected threshold is {float(overall['mean_selected_threshold']):.4f}. Configuration-search differences between rules are smaller than the variation across request-order seeds, so the ablation results should not be read as evidence that any single component is responsible for performance.
"""

existing_text = REPORT_PATH.read_text() if REPORT_PATH.exists() else ""

if REVISION_MARKER not in existing_text:
    with open(REPORT_PATH, "a") as file:
        file.write(revision_note)

print("Revision note present in:", REPORT_PATH)

## 14.11 Final Completeness Check

In [ ]:
required_outputs = [
    DIRS["results"] / "adaptive_configuration_search.csv",
    DIRS["results"] / "adaptive_validation_results.csv",
    DIRS["results"] / "adaptive_ablation_results.csv",
    DIRS["results"] / "adaptive_training_runs.csv",
    DIRS["results"] / "adaptive_cross_week_results.csv",
    DIRS["results"] / "adaptive_cross_week_summary.csv",
    DIRS["results"] / "adaptive_model_level_summary.csv",
    DIRS["results"] / "adaptive_overall_summary.csv",
    DIRS["results"] / "adaptive_threshold_results.csv",
    DIRS["results"] / "adaptive_threshold_diagnostic.csv",
    DIRS["configs"] / "adaptive_bee_configuration.json",
    DIRS["report"] / "adaptive_bee_report_section.md",
    DIRS["report"] / "adaptive_figure_captions.json",
]

missing_outputs = [str(path) for path in required_outputs if not path.exists()]

assert not missing_outputs, f"Missing outputs: {missing_outputs}"

print("Revised adaptive BEE stream is complete.")
print("Still to run once teammates' files exist: Section 11.3 (paired comparison with plain PPO and fixed BEE).")